In [49]:
# Imported Libraries
import nest_asyncio
nest_asyncio.apply()

from ib_async import IB, Stock, MarketOrder, util
from datetime import datetime, timedelta
import pandas as pd

In [50]:
# Connecting to the API
owned_stocks = []

try:
    ib.disconnect()
except NameError:
    pass

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=11)

train = True

In [ ]:
# Get The Data And Modify For The Model
if train:
    history_length = datetime.now() - timedelta(days=730)
    stock_info_dict = {}

    with open("data/S&P_500_Stocks.txt") as tickers:
        for ticker in tickers:
            ticker = ticker.strip()
            stock = Stock(ticker, "SMART", "USD")
            data = ib.reqHistoricalData(stock, endDateTime='', durationStr='4 Y',barSizeSetting='1 day', useRTH=True, whatToShow='ADJUSTED_LAST')
            pd_data = util.df(data)
            pd_data = pd_data.sort_values(by="date").dropna()

            pd_data["percentage_change"] = pd_data["close"].pct_change()
            pd_data["return_per_share"] = pd_data["close"].diff()

            pd_data["5_ma"] = pd_data["average"].rolling(5).mean()
            pd_data["25_ma"] = pd_data["average"].rolling(25).mean()
            pd_data["50_ma"] = pd_data["average"].rolling(50).mean()
            pd_data["200_ma"] = pd_data["average"].rolling(200).mean()

            ema_12 = pd_data["average"].ewm(12).mean()
            ema_26 = pd_data["average"].ewm(26).mean()

            pd_data["MACD"] = ema_12 - ema_26
            pd_data["Signal"] = pd_data["MACD"].rolling(9).mean()

            K = ((pd_data["close"] - min) / (max - min)) * 100

            pd_data["Stochastic Oscillator"] = K.rolling(3).mean()

            sma_20 = pd_data["average"].rolling(20).mean()
            std_20 = pd_data["average"].rolling(20).std()
            pd_data["Upper Band"] = sma_20 + 2 * std_20
            pd_data["Lower Band"] = sma_20 - 2 * std_20

            typical_price = (pd_data["high"] + pd_data["low"] + pd_data["close"]) / 3
            pd_data["VWAP"] = typical_price * pd_data["volume"] / pd_data["volume"]

            pd_data.drop(columns = ["open", "high", "low", "volume", "average", "barCount"], inplace=True)

            stock_info_dict[ticker] = pd_data

    print(stock_info_dict["NVDA"])

            date     close  percentage_change  return_per_share       5_ma  \
0     2022-09-12   14.4691                NaN               NaN        NaN   
1     2022-09-13   13.0985          -0.094726           -1.3706        NaN   
2     2022-09-14   13.0955          -0.000229           -0.0030        NaN   
3     2022-09-15   12.8970          -0.015158           -0.1985        NaN   
4     2022-09-16   13.1654           0.020811            0.2684   13.33533   
...          ...       ...                ...               ...        ...   
996   2026-09-01  217.4400          -0.015128           -3.3400  219.16880   
997   2026-09-02  224.4100           0.032055            6.9700  221.92280   
998   2026-09-03  228.4500           0.018003            4.0400  222.29000   
999   2026-09-04  230.3600           0.008361            1.9100  224.35580   
1000  2026-09-08  225.8200          -0.019708           -4.5400  226.03620   

          25_ma      50_ma      200_ma      MACD    Signal  \
0

In [52]:
# Training The Model
if train:
    0

In [53]:
# Saving The Model
if train:
    0

In [54]:
# Running The Model
if not train:
    0

In [55]:
# Process The Buys And Sells

In [56]:
# Disconnect From API
ib.disconnect()

'Disconnecting from 127.0.0.1:7497, 192 B sent in 9 messages, 97.8 kB received in 405 messages, session time 651 ms.'